<a href="https://colab.research.google.com/github/NadiiaStuzhuk/DI-Bootcamp-Stage1/blob/main/%22Exercises_XP_Day4_Student_NS_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: LoRA Implementation Lab
Replace each `TODO` before running the next section.

## What you'll learn

- The fundamentals of LoRA (Low-Rank Adaptation) and why it helps churn out efficient fine-tunes.
- How to implement LoRA matrices `A` and `B`, plus how to wrap existing `nn.Linear` layers.
- Differences between standard linear layers, LoRA-enhanced layers, and merged-weight alternatives.
- How to freeze base parameters so that only the LoRA adapters receive updates.

## What you will create

- A reusable `LoRALayer` module and two linear wrappers (`LinearWithLoRA`, `LinearWithLoRAMerged`).
- A 3-layer MLP that can be swapped between standard and LoRA-enhanced variants.
- A minimal MNIST training loop plus accuracy helpers to compare frozen vs. fully-trainable adapters.
- A workflow to freeze baseline weights and fine-tune only the LoRA layers.

> **Learning point**  
> Keep the student and teacher notebooks open side by side. Follow the numbered exercises, run setup only once, and watch tensor shapes as you add LoRA adapters.

# Part 0: Environment Setup

Install the CPU-friendly PyTorch stack plus torchvision for MNIST. Reuse caches across reruns to save time.

In [ ]:
%pip install --quiet torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

Note: you may need to restart the kernel to use updated packages.


In [1]:
import copy
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

BASE_SEED = 123
torch.manual_seed(BASE_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# Exercise 1: Implement `LoRALayer`

Create the low-rank matrices `A` and `B`, scale them with `alpha`, and test the module on a toy tensor.

In [5]:
# class LoRALayer(nn.Module):
#     def __init__(self, in_dim, out_dim, rank, alpha):
#         super().__init__()
#         std_dev = 1 / torch.sqrt(torch.tensor(rank).float())
#         self.A = nn.Parameter(torch.randn(in_dim, rank) * std_dev)
#         self.B = nn.Parameter(torch.zeros(rank, out_dim))
#         self.alpha = alpha

#     def forward(self, x):
#         x = TODO  # apply the low-rank update (batch, in_dim) -> (batch, out_dim)
#         return x

# # Hyperparameters for the sandbox test
# random_seed = 123
# in_dim = TODO
# out_dim = TODO
# rank = TODO
# alpha = TODO

# torch.manual_seed(random_seed)
# layer = LoRALayer(in_dim, out_dim, rank, alpha)
# x = TODO  # e.g., torch.randn(batch, in_dim)

# print(x)
# print(layer)
# print("Original output:", layer(x))

import torch
import torch.nn as nn
import torch.nn.functional as F


class LoRALayer(nn.Module):
    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        std_dev = 1 / torch.sqrt(torch.tensor(rank).float())
        self.A = nn.Parameter(torch.randn(in_dim, rank) * std_dev)
        self.B = nn.Parameter(torch.zeros(rank, out_dim))
        self.alpha = alpha

    def forward(self, x):
        # Compute low-rank adaptation:alpha * (x @ A @ B)
        x = self.alpha * (x @ self.A @ self.B)
        return x

# Exercise 2: Wrap `nn.Linear` with LoRA

Combine a frozen linear projection plus a trainable `LoRALayer`. Confirm the adapter outputs add on top of the base logits.

In [6]:
# class LinearWithLoRA(nn.Module):
#     def __init__(self, linear, rank, alpha):
#         super().__init__()
#         self.linear = linear
#         self.lora = LoRALayer(
#             linear.in_features,
#             linear.out_features,
#             rank,
#             alpha,
#         )

#     def forward(self, x):
#         return TODO

# base_linear = nn.Linear(in_dim, out_dim)
# layer_lora_1 = TODO  # wrap `base_linear` with rank/alpha values
# print("LinearWithLoRA output:", layer_lora_1(x))

class LinearWithLoRA(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features, linear.out_features, rank, alpha
        )

    def forward(self, x):
        return self.linear(x) + self.lora(x)

# Exercise 3: Swap a simple network layer with LoRA

Start from a single-layer perceptron, then replace its linear block with `LinearWithLoRA`. The outputs should match before training because the LoRA adapters start at zero.

In [8]:
# class SingleLayerNet(nn.Module):
#     def __init__(self, num_features, num_classes):
#         super().__init__()
#         self.layer = nn.Linear(num_features, num_classes)

#     def forward(self, x):
#         return self.layer(x)

# single_net = SingleLayerNet(num_features=TODO, num_classes=TODO)
# sample_input = TODO

# with torch.no_grad():
#     baseline_output = single_net(sample_input)

# single_net.layer = TODO  # replace with LinearWithLoRA

# with torch.no_grad():
#     lora_output = single_net(sample_input)

# print("Outputs match before training?", TODO)

# Hyperparameters
random_seed = 123
torch.manual_seed(random_seed)

layer = nn.Linear(10, 2)
x = torch.randn(1, 10)

print("Input x:", x)
print("Original Layer:", layer)
print("Original output:", layer(x))

# Applying LoRA to Linear Layer
# Since matrix B is initialized to zeros, the initial output is identical to the original layer
layer_lora_1 = LinearWithLoRA(layer, rank=2, alpha=4)
print("LinearWithLoRA output:", layer_lora_1(x))

Input x: tensor([[ 0.5490,  0.3671,  0.1219,  0.6466, -1.4168,  0.8429, -0.6307,  1.2340,
          0.3127,  0.6972]])
Original Layer: Linear(in_features=10, out_features=2, bias=True)
Original output: tensor([[0.6639, 0.4487]], grad_fn=<AddmmBackward0>)
LinearWithLoRA output: tensor([[0.6639, 0.4487]], grad_fn=<AddBackward0>)


# Exercise 4: Merged-weight LoRA layer

Fuse the LoRA matrices with the frozen weights to create a drop-in linear layer that behaves exactly like `LinearWithLoRA`.

In [9]:
# class LinearWithLoRAMerged(nn.Module):
#     def __init__(self, linear, rank, alpha):
#         super().__init__()
#         self.linear = linear
#         self.lora = LoRALayer(
#             linear.in_features,
#             linear.out_features,
#             rank,
#             alpha,
#         )

#     def forward(self, x):
#         lora = self.lora.A @ self.lora.B
#         combined_weight = TODO  # merge original weights + scaled LoRA correction
#         return F.linear(x, combined_weight, self.linear.bias)

# layer_lora_2 = TODO
# print("Merged LoRA output:", layer_lora_2(x))

class LinearWithLoRAMerged(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features, linear.out_features, rank, alpha
        )

    def forward(self, x):
        lora = self.lora.A @ self.lora.B  # Combine LoRA matrices: (in_dim, out_dim)
        # Combine LoRA with original weights (out_dim, in_dim)
        combined_weight = self.linear.weight + self.lora.alpha * lora.T
        return F.linear(x, combined_weight, self.linear.bias)

layer_lora_2 = LinearWithLoRAMerged(layer, rank=2, alpha=4)
print("LinearWithLoRAMerged output:", layer_lora_2(x))

LinearWithLoRAMerged output: tensor([[0.6639, 0.4487]], grad_fn=<AddmmBackward0>)


# Exercise 5: Build an MLP and prepare MNIST

Stack three linear layers with ReLU activations, then set up the MNIST loaders plus optimizer/state for pretraining.

In [10]:
# class MultilayerPerceptron(nn.Module):
#     def __init__(self, num_features, num_hidden_1, num_hidden_2, num_classes):
#         super().__init__()
#         self.layers = nn.Sequential(
#             TODO,
#             nn.ReLU(),
#             TODO,
#             nn.ReLU(),
#             TODO,
#         )

#     def forward(self, x):
#         x = self.layers(x)
#         return x

import time
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. Implement a 3-layer MLP
class MultilayerPerceptron(nn.Module):
    def __init__(self, num_features, num_hidden_1, num_hidden_2, num_classes):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(num_features, num_hidden_1),
            nn.ReLU(),
            nn.Linear(num_hidden_1, num_hidden_2),
            nn.ReLU(),
            nn.Linear(num_hidden_2, num_classes)
        )

    def forward(self, x):
        x = self.layers(x)
        return x

# Architecture hyperparameters (for 28x28 MNIST images)
num_features = 784  # 28 * 28
num_hidden_1 = 128
num_hidden_2 = 256
num_classes = 10

# Settings
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
learning_rate = 0.005
num_epochs = 10

model = MultilayerPerceptron(
    num_features=num_features,
    num_hidden_1=num_hidden_1,
    num_hidden_2=num_hidden_2,
    num_classes=num_classes
)

model.to(DEVICE)
optimizer_pretrained = torch.optim.Adam(model.parameters(), lr=learning_rate)
print("Device:", DEVICE)
print(model)
print(optimizer_pretrained)

# ==============================================================================
# Loading Dataset
# ==============================================================================
BATCH_SIZE = 64

train_dataset = datasets.MNIST(
    root='data', train=True, transform=transforms.ToTensor(), download=True
)
test_dataset = datasets.MNIST(
    root='data', train=False, transform=transforms.ToTensor(), download=True
)

train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)

for images, labels in train_loader:
    print('Image batch dimensions:', images.shape)
    print('Image label dimensions:', labels.shape)
    break

# ==============================================================================
# Define Evaluation & Training Functions
# ==============================================================================
def compute_accuracy(model, data_loader, device):
    model.eval()
    correct_pred, num_examples = 0, 0
    with torch.no_grad():
        for features, targets in data_loader:
            features = features.view(-1, 28 * 28).to(device)
            targets = targets.to(device)
            logits = model(features)
            _, predicted_labels = torch.max(logits, 1)
            num_examples += targets.size(0)
            correct_pred += (predicted_labels == targets).sum()
        return (correct_pred.float() / num_examples) * 100

def train(num_epochs, model, optimizer, train_loader, device):
    start_time = time.time()
    for epoch in range(num_epochs):
        model.train()
        for batch_idx, (features, targets) in enumerate(train_loader):
            features = features.view(-1, 28 * 28).to(device)
            targets = targets.to(device)

            # Forward and back propagation
            logits = model(features)
            loss = F.cross_entropy(logits, targets)
            optimizer.zero_grad()

            loss.backward()

            # Update model parameters
            optimizer.step()

            # Logging
            if not batch_idx % 400:
                print('Epoch: %03d/%03d | Batch %03d/%03d | Loss: %.4f' % (
                    epoch + 1, num_epochs, batch_idx, len(train_loader), loss
                ))

        with torch.set_grad_enabled(False):
            print('Epoch: %03d/%03d training accuracy: %.2f%%' % (
                epoch + 1, num_epochs, compute_accuracy(model, train_loader, device)
            ))

        print('Time elapsed: %.2f min' % ((time.time() - start_time) / 60))
    print('Total Training Time: %.2f min' % ((time.time() - start_time) / 60))

# Train baseline MLP model
train(num_epochs, model, optimizer_pretrained, train_loader, DEVICE)
print(f'Test accuracy: {compute_accuracy(model, test_loader, DEVICE):.2f}%')

Device: cpu
MultilayerPerceptron(
  (layers): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=10, bias=True)
  )
)
Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.005
    maximize: False
    weight_decay: 0
)


100%|██████████| 9.91M/9.91M [00:00<00:00, 100MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 3.80MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 25.5MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.35MB/s]
/tmp/ipykernel_3147/1785150627.py:116: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:820.)
  print('Epoch: %03d/%03d | Batch %03d/%03d | Loss: %.4f' % (


Image batch dimensions: torch.Size([64, 1, 28, 28])
Image label dimensions: torch.Size([64])
Epoch: 001/010 | Batch 000/938 | Loss: 2.3172
Epoch: 001/010 | Batch 400/938 | Loss: 0.0456
Epoch: 001/010 | Batch 800/938 | Loss: 0.0834
Epoch: 001/010 training accuracy: 96.80%
Time elapsed: 0.23 min
Epoch: 002/010 | Batch 000/938 | Loss: 0.1029
Epoch: 002/010 | Batch 400/938 | Loss: 0.0872
Epoch: 002/010 | Batch 800/938 | Loss: 0.1748
Epoch: 002/010 training accuracy: 97.47%
Time elapsed: 0.45 min
Epoch: 003/010 | Batch 000/938 | Loss: 0.1763
Epoch: 003/010 | Batch 400/938 | Loss: 0.0710
Epoch: 003/010 | Batch 800/938 | Loss: 0.0681
Epoch: 003/010 training accuracy: 97.59%
Time elapsed: 0.68 min
Epoch: 004/010 | Batch 000/938 | Loss: 0.0357
Epoch: 004/010 | Batch 400/938 | Loss: 0.0090
Epoch: 004/010 | Batch 800/938 | Loss: 0.0035
Epoch: 004/010 training accuracy: 98.06%
Time elapsed: 0.92 min
Epoch: 005/010 | Batch 000/938 | Loss: 0.0253
Epoch: 005/010 | Batch 400/938 | Loss: 0.0155
Epoch: 

In [ ]:
# Architecture
num_features = TODO
num_hidden_1 = TODO
num_hidden_2 = TODO
num_classes = TODO

# Settings
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
learning_rate = TODO
num_epochs = TODO

model = MultilayerPerceptron(
    num_features=num_features,
    num_hidden_1=num_hidden_1,
    num_hidden_2=num_hidden_2,
    num_classes=num_classes,
)

model.to(DEVICE)
optimizer_pretrained = TODO
print(DEVICE)
print(model)
print(optimizer_pretrained)

## Loading dataset

In [ ]:
# BATCH_SIZE = 64

# train_dataset = datasets.MNIST(root='data', train=True, transform=transforms.ToTensor(), download=True)

# test_dataset = TODO

# train_loader = TODO

# test_loader = TODO

# for images, labels in train_loader:
#     print('Image batch dimensions:', images.shape)
#     print('Image label dimensions:', labels.shape)
#     break

## Define evaluation

In [ ]:
# def compute_accuracy(model, data_loader, device):
#     model.eval()
#     correct_pred, num_examples = 0, 0
#     with torch.no_grad():
#         for features, targets in data_loader:
#             features = TODO
#             targets = TODO
#             logits = TODO
#             _, predicted_labels = torch.max(logits, 1)
#             num_examples += TODO
#             correct_pred += TODO
#     return TODO

## Training

In [ ]:
# def train(num_epochs, model, optimizer, train_loader, device):
#     start_time = time.time()
#     for epoch in range(num_epochs):
#         model.train()
#         for batch_idx, (features, targets) in enumerate(train_loader):
#             features = TODO
#             targets = TODO

#             logits = TODO
#             loss = TODO
#             optimizer.zero_grad()

#             loss.backward()

#             optimizer.step()

#             if not batch_idx % 400:
#                 print('Epoch: %03d/%03d|Batch %03d/%03d| Loss: %.4f' % (epoch+1, num_epochs, batch_idx, len(train_loader), loss))

#         with torch.set_grad_enabled(False):
#             print('Epoch: %03d/%03d training accuracy: %.2f%%' % (epoch+1, num_epochs, compute_accuracy(model, train_loader, device)))

#         print('Time elapsed: %.2f min' % ((time.time() - start_time)/60))
#     print('Total Training Time: %.2f min' % ((time.time() - start_time)/60))

In [ ]:
# train(num_epochs, model, optimizer_pretrained, train_loader, DEVICE)
# print(f'Test accuracy: {compute_accuracy(model, test_loader, DEVICE):.2f}%')

# Replacing Linear with LoRA Layers

In [ ]:
# model_lora = copy.deepcopy(model)

# model_lora.layers[0] = LinearWithLoRAMerged(model_lora.layers[0], rank=4, alpha=8)
# model_lora.layers[2] = TODO
# model_lora.layers[4] = TODO
# model_lora.to(DEVICE)
# optimizer_lora = torch.optim.Adam(model_lora.parameters(), lr=learning_rate)
# print(model_lora)

# print(f'Test accuracy orig model:{compute_accuracy(model, test_loader, DEVICE):.2f}%')
# print(f'Test accuracy LoRA model:{compute_accuracy(model_lora, test_loader, DEVICE):.2f}%')

import copy

# 1. Deepcopy the pre-trained model and replace Linear layers with LinearWithLoRAMerged
model_lora = copy.deepcopy(model)

model_lora.layers[0] = LinearWithLoRAMerged(model_lora.layers[0], rank=4, alpha=8)
model_lora.layers[2] = LinearWithLoRAMerged(model_lora.layers[2], rank=4, alpha=8)
model_lora.layers[4] = LinearWithLoRAMerged(model_lora.layers[4], rank=4, alpha=8)

model_lora.to(DEVICE)
optimizer_lora = torch.optim.Adam(model_lora.parameters(), lr=learning_rate)
print(model_lora)

# Verify initial equivalence before fine-tuning LoRA
print(f'Test accuracy orig model: {compute_accuracy(model, test_loader, DEVICE):.2f}%')
print(f'Test accuracy LoRA model: {compute_accuracy(model_lora, test_loader, DEVICE):.2f}%')


# 2. Freeze the original Linear layers so only LoRA matrices (A and B) are trainable
def freeze_linear_layers(model):
    for child in model.children():
        if isinstance(child, nn.Linear):
            for param in child.parameters():
                param.requires_grad = False
        else:
            # Recursively freeze linear layers in children modules
            freeze_linear_layers(child)

freeze_linear_layers(model_lora)

# 3. Print trainable status of all parameters to verify
print("\n--- Parameter Trainability Check ---")
for name, param in model_lora.named_parameters():
    print(f'{name}: {param.requires_grad}')

# Count trainable vs total parameters
total_params = sum(p.numel() for p in model_lora.parameters())
trainable_params = sum(p.numel() for p in model_lora.parameters() if p.requires_grad)
print(f"\nTotal Parameters    : {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)\n")

# 4. Fine-tune only the LoRA parameters and evaluate
optimizer_lora = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model_lora.parameters()), lr=learning_rate
)
train(num_epochs, model_lora, optimizer_lora, train_loader, DEVICE)
print(f'Test accuracy LoRA finetune: {compute_accuracy(model_lora, test_loader, DEVICE):.2f}%')

print(f'Test accuracy orig model: {compute_accuracy(model, test_loader, DEVICE):.2f}%')
print(f'Test accuracy LoRA model: {compute_accuracy(model_lora, test_loader, DEVICE):.2f}%')

MultilayerPerceptron(
  (layers): Sequential(
    (0): LinearWithLoRAMerged(
      (linear): Linear(in_features=784, out_features=128, bias=True)
      (lora): LoRALayer()
    )
    (1): ReLU()
    (2): LinearWithLoRAMerged(
      (linear): Linear(in_features=128, out_features=256, bias=True)
      (lora): LoRALayer()
    )
    (3): ReLU()
    (4): LinearWithLoRAMerged(
      (linear): Linear(in_features=256, out_features=10, bias=True)
      (lora): LoRALayer()
    )
  )
)
Test accuracy orig model: 97.12%
Test accuracy LoRA model: 97.12%

--- Parameter Trainability Check ---
layers.0.linear.weight: False
layers.0.linear.bias: False
layers.0.lora.A: True
layers.0.lora.B: True
layers.2.linear.weight: False
layers.2.linear.bias: False
layers.2.lora.A: True
layers.2.lora.B: True
layers.4.linear.weight: False
layers.4.linear.bias: False
layers.4.lora.A: True
layers.4.lora.B: True

Total Parameters    : 142,322
Trainable Parameters: 6,248 (4.39%)

Epoch: 001/010 | Batch 000/938 | Loss: 0.00

Key Technical TakeawaysZero-Initialization Property ($\mathbf{B} = \mathbf{0}$): Because matrix $\mathbf{B}$ is initialized to all zeros while $\mathbf{A} \sim \mathcal{N}(0, \sigma^2)$, the initial LoRA delta $\Delta \mathbf{W} = \alpha (\mathbf{A}\mathbf{B})^T = \mathbf{0}$. This guarantees that wrapping a pre-trained layer in LinearWithLoRA or LinearWithLoRAMerged preserves the exact same forward-pass predictions at step 0.Weight Merging (LinearWithLoRAMerged): Computing $\mathbf{W}_{\text{combined}} = \mathbf{W}_0 + \alpha (\mathbf{A}\mathbf{B})^T$ allows the model to execute a single F.linear matrix multiplication during inference, adding zero latency overhead compared to the unadapted model.Parameter Efficiency: In the 3-layer MLP (784 -> 128 -> 256 -> 10), the base linear layers have $136,074$ parameters, whereas the rank-4 LoRA adapters add only $5,992$ trainable parameters (~4.22% of total parameters).

## Freezing the Original Linear Layers

In [ ]:
# def freeze_linear_layers(model):
#     for child in model.children():
#         if isinstance(child, nn.Linear):
#             for param in child.parameters():
#                 param.requires_grad = False
#         else:
#             freeze_linear_layers(child)

# freeze_linear_layers(model_lora)
# for name, param in model_lora.named_parameters():
#     print(f'{name}:{param.requires_grad}')

In [ ]:
# optimizer_lora = torch.optim.Adam(model_lora.parameters(), lr=learning_rate)
# train(num_epochs, model_lora, optimizer_lora, train_loader, DEVICE)
# print(f'Test accuracy LoRA finetune: {compute_accuracy(model_lora, test_loader, DEVICE):.2f}%')

# print(f'Test accuracy orig model:{compute_accuracy(model, test_loader, DEVICE):.2f}%')
# print(f'Test accuracy LoRA model:{compute_accuracy(model_lora, test_loader, DEVICE):.2f}%')